# SLM Model Build with DSA **course**

### Introduction
This notebook guides you through building a Small Language Model (SLM) for Data Structures and Algorithms (DSA) topics, incorporating Retrieval-Augmented Generation (RAG) to enhance its knowledge and prevent hallucination. We fine-tune a pre-trained `TinyLlama` model using LoRA and integrate a FAISS-based RAG system for contextual understanding.

### STEP 1: Environment Setup
We begin by installing necessary Python libraries. `transformers`, `accelerate`, `peft`, `datasets`, `sentence-transformers`, and `faiss-cpu` are crucial for model loading, efficient fine-tuning (LoRA), data handling, embedding generation, and efficient similarity search respectively. We uninstall `bitsandbytes` to ensure compatibility or avoid potential conflicts.

In [1]:
print("Installing required libraries...")

!pip uninstall -y bitsandbytes
!pip install -q --upgrade transformers accelerate peft datasets sentence-transformers faiss-cpu

import torch
import json
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model   # IMPORTANT
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np



Installing required libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [2]:
import importlib.util
print(importlib.util.find_spec("bitsandbytes"))


None


### STEP 2: Data Loading
We load a `dsa_training.json` file, which contains our domain-specific training data. This data is essential for fine-tuning the base language model to understand and generate content relevant to Data Structures and Algorithms. The JSON format allows for structured storage of text examples.

In [ ]:
from google.colab import files
print("Upload your dsa_training.json file")
uploaded = files.upload()

TRAINING_FILE = list(uploaded.keys())[0]
with open(TRAINING_FILE, "r") as f:
    training_data = json.load(f)

print(f"Loaded {len(training_data)} training examples")


Upload your dsa_training.json file


Saving dsa_training.json to dsa_training.json
Loaded 380 training examples


### STEP 3: Model and Tokenizer Loading
We utilize `TinyLlama/TinyLlama-1.1B-Chat-v1.0` as our base model. This is a compact yet capable language model suitable for fine-tuning on domain-specific tasks, especially in environments with limited resources. The `AutoTokenizer` is loaded to ensure proper tokenization consistent with the pre-trained model's vocabulary and formatting. `torch_dtype=torch.float16` and `device_map='auto'` are used for memory efficiency and optimal device utilization.

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

print("Base model loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Base model loaded


### STEP 4: Configuring LoRA
Low-Rank Adaptation (LoRA) is a parameter-efficient fine-tuning technique. Instead of fine-tuning all model parameters, LoRA injects small, trainable matrices into specific layers (like query, key, value, and output projections in attention mechanisms). This significantly reduces the number of trainable parameters, making fine-tuning faster and more memory-efficient while achieving comparable performance to full fine-tuning. We configure `r` (rank) and `lora_alpha` to control the adaptation's expressiveness, and `lora_dropout` for regularization.

In [ ]:
from peft import LoraConfig, get_peft_model

print("="*70)
print("STEP 4: Configuring LoRA")
print("="*70)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nLoRA attached successfully")


STEP 4: Configuring LoRA
trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079

LoRA attached successfully


### STEP 5: Preparing Training Dataset
The loaded JSON data is converted into a Hugging Face `Dataset` object. Each example is formatted, and then tokenized using the loaded `TinyLlama` tokenizer. Tokenization breaks down text into numerical tokens that the model can process. We set `truncation=True` and `padding='max_length'` to handle variable text lengths, and create `labels` by copying `input_ids` for causal language modeling.

In [ ]:
print("="*70)
print("STEP 5: Preparing Training Dataset")
print("="*70)

# i change GPT like multi response
# def format_dsa_example(example):
#     return {"text": example["text"]}

def format_dsa_example(example):
    system_prompt = (
        "You are a senior DSA tutor.\n"
        f"Student emotion: {example.get('emotion', 'neutral')}.\n"
        "Teaching rules:\n"
        "- Explain step by step\n"
        "- Never leave sections empty\n"
        "- If code is shown, explain it\n"
        "- Adjust depth based on emotion\n\n"
    )
    return {"text": system_prompt + example["text"]}


# Convert to HuggingFace dataset
dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_dsa_example)

print(f"Dataset size: {len(dataset)} examples")

# Tokenize
def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=dataset.column_names,
    batched=True
)

print("✓ Dataset tokenized and ready")


STEP 5: Preparing Training Dataset


Map:   0%|          | 0/380 [00:00<?, ? examples/s]

Dataset size: 380 examples
Tokenizing dataset...


Map:   0%|          | 0/380 [00:00<?, ? examples/s]

✓ Dataset tokenized and ready


### STEP 6: Training Configuration
`TrainingArguments` defines all parameters for the fine-tuning process, such as `num_train_epochs` (number of passes over the dataset), `per_device_train_batch_size` (examples processed per GPU/CPU), `gradient_accumulation_steps` (accumulating gradients over multiple steps to simulate a larger batch size), `learning_rate`, and `fp16` (for mixed-precision training). `DataCollatorForLanguageModeling` is used to prepare batches of data for training, specifically for causal language modeling where the model predicts the next token.

In [ ]:
print("="*70)
print("STEP 6: Training Configuration")
print("="*70)

training_args = TrainingArguments(
    output_dir="./dsa-slm-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    warmup_steps=10,
    logging_dir="./logs",
    report_to="none",
    remove_unused_columns=False,
    optim="adamw_torch"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("✓ Training configuration ready")


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


STEP 6: Training Configuration
✓ Training configuration ready


### STEP 7: Creating Trainer
The `Trainer` class from Hugging Face is a high-level API for training models. It abstracts away the training loop details, making it easier to fine-tune models. We pass the `model` (our LoRA-adapted `TinyLlama`), `training_args`, `tokenized_dataset`, and `data_collator` to initialize the trainer, which will manage the training process.

In [ ]:
print("="*70)
print("STEP 7: Creating Trainer")
print("="*70)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("✓ Trainer created successfully")


STEP 7: Creating Trainer
✓ Trainer created successfully


### STEP 8: Starting Training
This step initiates the fine-tuning process by calling `trainer.train()`. During training, the model learns from the DSA-specific data, adjusting its weights (specifically, the LoRA matrices) to better understand and generate content related to data structures and algorithms. The process can be time-consuming, depending on the dataset size and hardware.

In [ ]:
print("="*70)
print("STEP 8: STARTING TRAINING")
print("="*70)
print("⏰ This will take 2–3 hours depending on dataset size")
print("☕ Keep Colab tab open to avoid disconnect")
print("="*70 + "\n")

trainer.train()

print("\n" + "="*70)
print("✓ TRAINING COMPLETE!")
print("="*70)


STEP 8: STARTING TRAINING
⏰ This will take 2–3 hours depending on dataset size
☕ Keep Colab tab open to avoid disconnect



Step,Training Loss
10,1.977979
20,1.180597
30,0.372092
40,0.172035
50,0.140581
60,0.144491
70,0.134082
80,0.117534
90,0.124682
100,0.110305



✓ TRAINING COMPLETE!


### STEP 9: Saving Your DSA SLM
After training, the fine-tuned model and its tokenizer are saved to a local directory (`./my-dsa-slm`). Saving the model's adapter weights (from LoRA) and the tokenizer ensures that the fine-tuning can be reloaded and used for inference or further training without needing the original base model weights to be modified directly. This makes the fine-tuned model portable.

In [ ]:
print("="*70)
print("STEP 9: Saving Your DSA SLM")
print("="*70)

output_dir = "./my-dsa-slm"

# Save LoRA fine-tuned model
trainer.save_model(output_dir)

# Save tokenizer
tokenizer.save_pretrained(output_dir)

print(f"✓ Model and tokenizer saved at: {output_dir}")


STEP 9: Saving Your DSA SLM
✓ Model and tokenizer saved at: ./my-dsa-slm


### STEP 10: Building RAG System
A Retrieval-Augmented Generation (RAG) system is built to ground the SLM's answers in specific, provided content, reducing hallucinations.

1.  **`SentenceTransformer` (`all-MiniLM-L6-v2`):** This model is used to convert textual documents into numerical vector embeddings. These embeddings capture the semantic meaning of the text.
2.  **FAISS (`faiss.IndexFlatL2`):** This library provides efficient similarity search. We create a FAISS index from the embeddings of our training documents. When a query comes, its embedding is compared against the indexed document embeddings to find the most relevant documents (`top_k`).

The RAG system stores original documents and their embeddings, allowing for fast retrieval of relevant context based on user queries. The built RAG system is then serialized using `pickle` and saved for later use.

In [ ]:
print("="*70)
print("STEP 10: Building RAG System")
print("="*70)
print("This prevents hallucination by grounding answers in YOUR content\n")

class DSA_RAG:
    def __init__(self):
        self.encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.documents = []
        self.index = None

    def build_index(self, training_data):
        # Store documents
        self.documents = [
            {
                "content": item["text"],
                "type": item.get("type", "general"),
                "module": item.get("module", "N/A")
            }
            for item in training_data
        ]

        # Create embeddings
        texts = [doc["content"] for doc in self.documents]
        embeddings = self.encoder.encode(texts, show_progress_bar=True)

        # Build FAISS index
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings.astype("float32"))

        print(f"✓ RAG index built with {len(self.documents)} documents")

    def retrieve(self, query, top_k=3):
        query_embedding = self.encoder.encode([query])
        distances, indices = self.index.search(
            query_embedding.astype("float32"),
            top_k
        )
        results = [self.documents[idx] for idx in indices[0]]
        return results


# Build RAG
rag = DSA_RAG()
rag.build_index(training_data)

# Save RAG system
import pickle
with open("dsa_rag.pkl", "wb") as f:
    pickle.dump(rag, f)

print("✓ RAG system built and saved as dsa_rag.pkl")


STEP 10: Building RAG System
This prevents hallucination by grounding answers in YOUR content



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✓ RAG index built with 380 documents
✓ RAG system built and saved as dsa_rag.pkl


### STEP 11: Testing Your DSA SLM
This section demonstrates how to use the fine-tuned SLM with and without RAG. The `test_dsa_slm` function takes a `question` and an optional `use_rag` flag.

*   **With RAG (`use_rag=True`):** The system first retrieves relevant documents using the FAISS index based on the query. These retrieved documents are then prepended to the prompt as `context`, guiding the SLM to generate an answer grounded in the provided information.
*   **Without RAG (`use_rag=False`):** The SLM generates an answer solely based on its fine-tuned knowledge.

`model.generate()` is used for text generation, with parameters like `max_new_tokens`, `temperature`, and `top_p` controlling the length, creativity, and diversity of the generated response.

In [ ]:
print("="*70)
print("STEP 11: Testing Your DSA SLM")
print("="*70 + "\n")

def test_dsa_slm(question, use_rag=True):
    """
    Test your trained SLM with or without RAG
    """
    if use_rag:
        # Retrieve relevant context from RAG
        docs = rag.retrieve(question, top_k=2)
        context = "\n\n".join([d["content"][:500] for d in docs])

        prompt = f"""Based on this DSA course content:
{context}

Question: {question}
Answer:"""
    else:
        prompt = f"Question: {question}\nAnswer:"

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.1,      # low = factual answers
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("Answer:")[-1].strip()
    return answer


# Sample test questions
test_questions = [
    "What is Big-O notation?",
    "Explain binary search",
    "When should I use a hash map?",
    "What is the difference between array and vector?",
    "Explain recursion with example"
]

print("Testing with sample DSA questions:\n")

for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"TEST {i}: {q}")
    print("="*70)
    answer = test_dsa_slm(q)
    print(answer)


STEP 11: Testing Your DSA SLM

Testing with sample DSA questions:


TEST 1: What is Big-O notation?
Big-O notation represents the order of complexity of an algorithm as a function of its input size. It helps in optimizing code by choosing the right data structure and algorithm.

ACKNOWLEDGEMENT:
It's important to understand this concept to implement it effectively.

RECAP:
• Understand the problem this concept solves
• Learn how it works internally
• Apply in real problems with support

ASCII DIAGRAM:
Input  ->  Processing  ->  Output

CODE (C++):
```cpp
// Example implementation
#include <iostream>
using namespace std;

int main() {
    // Big-O Notation implementation
    cout << "Hello Big-O Notation" << endl;
    return 0;
}
```

TIPS & ADVICE:
• Repeat implementation until mastery
• Write code example & explain it

TEST 2: Explain binary search
Yes, it's implemented in DSA with code & documentation

ACKNOWLEDGEMENT:
It's completely okay to ask this question. DSA team will answer i

### STEP 12: Download Your Model
To make the fine-tuned SLM and the RAG system portable, they are zipped into `my-dsa-slm.zip` and `dsa-rag.zip` respectively. These compressed files can then be easily downloaded, allowing you to deploy or share your custom DSA SLM and its associated knowledge base.

In [ ]:
print("="*70)
print("STEP 12: Download Your Model")
print("="*70)

# Zip the trained SLM model
!zip -r my-dsa-slm.zip ./my-dsa-slm

# Zip the RAG system
!zip dsa-rag.zip dsa_rag.pkl

print("\n✓ Files zipped successfully")
print("Preparing for download...")

from google.colab import files
files.download("my-dsa-slm.zip")
files.download("dsa-rag.zip")

print("\n✓ Download complete!")


STEP 12: Download Your Model
  adding: my-dsa-slm/ (stored 0%)
  adding: my-dsa-slm/tokenizer_config.json (deflated 46%)
  adding: my-dsa-slm/adapter_config.json (deflated 57%)
  adding: my-dsa-slm/README.md (deflated 66%)
  adding: my-dsa-slm/chat_template.jinja (deflated 60%)
  adding: my-dsa-slm/training_args.bin (deflated 53%)
  adding: my-dsa-slm/tokenizer.json (deflated 85%)
  adding: my-dsa-slm/adapter_model.safetensors (deflated 8%)
  adding: dsa_rag.pkl (deflated 9%)

✓ Files zipped successfully
Preparing for download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Download complete!


In [ ]:
!pip install -q huggingface_hub


In [ ]:
!huggingface-cli login


/bin/bash: line 1: huggingface-cli: command not found


### STEP 13: Upload to Hugging Face Hub
This final step allows you to share your fine-tuned model, tokenizer, and the RAG system with the community via the Hugging Face Hub. After logging in with `huggingface-cli login`, the `model.push_to_hub()` and `tokenizer.push_to_hub()` methods are used to upload the model and tokenizer. The `dsa_rag.pkl` file is also uploaded using `api.upload_file()`. This makes your work accessible, shareable, and reproducible.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

model.push_to_hub("rohanrs09/dsa-slm")
tokenizer.push_to_hub("rohanrs09/dsa-slm")


## Summary:

### Data Analysis Key Findings

*   **Unfulfilled Prerequisites:** Across all subtasks, the primary and recurring issue was the non-completion or non-persistence of prerequisite steps, specifically the training and saving of the fine-tuned model, tokenizer, and the construction of the RAG system (`dsa_rag.pkl`, `my-dsa-slm`, `dsa-slm-output`). These components were consistently reported as "not found" or "not initialized."
*   **Dependency on Manual User Action:** The agent repeatedly identified that the system's core components (fine-tuned model, tokenizer, RAG) were not available. Its resolution strategy involved adding markdown cells to the notebook, providing detailed instructions to the user to manually execute all prerequisite code cells (from 'STEP 1: Environment Setup' to 'STEP 10: Building RAG System').
*   **Failed RAG Visualization:** The task to visualize RAG-retrieved contexts could not be completed because the `rag` system was never successfully loaded. This failure was compounded by issues such as missing `faiss-cpu` library, incorrect path interpretation (`HFValidationError`), and the absence of saved model/RAG files (`OSError`).
*   **Failed Training Log Analysis:** Similarly, the task to analyze and plot training loss failed because the `dsa-slm-output` directory and the `trainer_state.json` file were not found, indicating that the model training process had not been executed or its output was not accessible.
*   **Agent's Execution Limitations:** The agent demonstrated an inability to programmatically ensure or verify the successful execution of complex, multi-step prerequisites (like model training which can take significant time) within the environment. It resorted to iterative instructional markdown generation.

### Insights or Next Steps

*   **Improve Prerequisite Handling:** Implement a more robust mechanism to programmatically check for the existence and validity of essential components (fine-tuned model, tokenizer, RAG system, training logs). If missing, guide the user to a dedicated prerequisite setup block that can be run reliably, or consider automated execution if the environment allows.
*   **Enhanced State Management:** Develop clearer state management for long-running processes like model training. The system should ideally persist or be able to re-load the state of its fine-tuned models and RAG index across sessions or agent turns to avoid repeated prerequisite failures.
